# Session 3: Tool Use, Agents & Orchestration

**Great American Insurance Group — AI Developer Training**

In Session 2 you built a RAG pipeline: chunk, embed, store, retrieve, augment, generate. Every step ran in a fixed order you wrote yourself. Today you'll take that same pipeline and turn its pieces into **tools** — functions the model can choose to call, with what arguments, and in what order. You'll build the loop that lets it do that, then use it to produce a structured underwriting decision.

Throughout the notebook you'll see:
- 📝 **Exercise** / **Try It Yourself** — write or modify something yourself
- 💬 **Reflection** — a question to think about or discuss
- 💡 **Key Takeaway** — the main point to walk away with

Same rule as Sessions 1–2: all LLM calls go through `get_client().generate(...)` / `get_client().generate_structured(...)` / `get_client().generate_with_tools(...)`, and all embedding calls go through `get_client().embed(...)`. Never a raw `boto3`/`openai` SDK call.

This session reuses the same 4 synthetic underwriting PDFs from Session 2 (`../session-2-rag-retrieval/docs/`) — no new documents, no new setup.

## Section 1 — Recap & rebuild the pipeline

**Goal:** Reconnect to where Session 2 left off and have a working retrieval index ready to wrap in tools.

Quick recap of Session 2's RAG pipeline:
- **Offline (indexing):** documents → chunk (semantic, meaning-based boundaries) → embed → store in Chroma
- **Online (query time):** question → embed → retrieve top-K similar chunks → augment the prompt → generate an answer

And Session 1's structured-output pattern: define a Pydantic model, pass it to `generate_structured(...)`, get back a validated object instead of hand-parsed JSON.

The cell below rebuilds Session 2's index from scratch, using the same semantic-chunking code (its meaning-based boundaries retrieve better than fixed-size chunking, which is why it's the version carried forward here) — this is a fast, mostly-provided rebuild, not a new teaching moment.

### Guided Demo — rebuild the index

In [ ]:
from shared.llm import get_client

client = get_client()

In [ ]:
from pathlib import Path

from pypdf import PdfReader

# Same 4 PDFs as Session 2 — read straight from that session's docs/ folder rather
# than duplicating them here.
DOCS_DIR = Path("../session-2-rag-retrieval/docs")

DOC_IDS = ["GL-GUIDELINES", "PROPERTY-APPETITE", "CYBER-EXCLUSIONS", "WORKCOMP-CLASSIFICATION"]

def load_pdf_doc(path: Path) -> dict:
    """Extract title + body text from one of our underwriting-manual PDFs."""
    reader = PdfReader(path)
    raw_text = "\n".join(page.extract_text() for page in reader.pages)
    lines = [line.strip() for line in raw_text.splitlines() if line.strip()]

    title = lines[1]
    body = " ".join(lines[3:])
    return {"title": title, "text": body}

UNDERWRITING_DOCS = {}
for doc_id in DOC_IDS:
    filename = doc_id.lower() + ".pdf"
    UNDERWRITING_DOCS[doc_id] = load_pdf_doc(DOCS_DIR / filename)

print(f"Loaded {len(UNDERWRITING_DOCS)} documents.")

In [ ]:
import math
import re

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b)

def semantic_chunk(text: str, percentile: float = 40) -> list[str]:
    """Group sentences into chunks based on embedding similarity, splitting where
    similarity drops below the given percentile of this document's own
    sentence-to-sentence similarity distribution — not a fixed size or a fixed
    absolute similarity value.
    """
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    if len(sentences) <= 1:
        return sentences

    embeddings = client.embed(sentences)
    similarities = [
        cosine_similarity(embeddings[i - 1], embeddings[i])
        for i in range(1, len(sentences))
    ]

    threshold = sorted(similarities)[int(len(similarities) * percentile / 100)]

    chunks = []
    current_chunk = [sentences[0]]
    for i, similarity in enumerate(similarities, start=1):
        if similarity >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
    chunks.append(" ".join(current_chunk))
    return chunks

In [ ]:
import chromadb

ALL_CHUNKS = []
for doc_id, doc in UNDERWRITING_DOCS.items():
    for i, chunk in enumerate(semantic_chunk(doc["text"])):
        ALL_CHUNKS.append({"text": chunk, "source": doc_id, "chunk_index": i})

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("underwriting_docs")

chunk_texts = [c["text"] for c in ALL_CHUNKS]
chunk_embeddings = client.embed(chunk_texts)

collection.add(
    ids=[f"{c['source']}-{c['chunk_index']}" for c in ALL_CHUNKS],
    embeddings=chunk_embeddings,
    documents=chunk_texts,
    metadatas=[{"source": c["source"], "chunk_index": c["chunk_index"]} for c in ALL_CHUNKS],
)

print("Collection count:", collection.count())

&nbsp;

### 🔧 Try It Yourself

Confirm the collection count above matches what you saw at the end of Session 2 (same 4 documents, same semantic chunking).

💬 **Discuss with your group:** of everything built in Session 2 — chunking, embedding, storing, retrieving, augmenting, generating — which of those steps should an LLM be allowed to *decide* to run, and which should just always run no matter what?

&nbsp;

💡 **Key Takeaway** — An agent's tools are just the application's existing functions — the RAG pipeline you already built is 90% of the toolbox.

---

## Section 2 — Creating tools

**Goal:** Understand what makes a Python function an LLM tool, and test each one in isolation before any agent exists.

A tool is three things: a **name**, a **description** the model reads to decide when to call it, and an **input schema** describing its arguments. The function body itself can be anything — another LLM call, a database query, or plain deterministic logic. Below we build three tools out of pieces you already have, one of each kind.

### Variant 1 — `search_documents` (RAG-backed)

Wraps Session 2's `retrieve()` — the only new work is reshaping Chroma's raw, parallel-list output into a clean list of hit dicts.

In [ ]:
def retrieve(query: str, k: int = 3):
    query_embedding = client.embed([query])[0]
    return collection.query(query_embeddings=[query_embedding], n_results=k)

📝 **Exercise 2.1** — `retrieve()` returns Chroma's native shape: `results["documents"][0]`, `results["metadatas"][0]`, and `results["distances"][0]` are three parallel lists (all length `k`). Fill in `search_documents` below so it zips them together into one list of `{"source", "chunk_index", "text", "distance"}` dicts — this reshaped output is what the model will actually see as the tool result.

In [ ]:
def search_documents(query: str, k: int = 3) -> list[dict]:
    """Search the underwriting guideline manuals for text relevant to `query`."""
    results = retrieve(query, k=k)

    hits = []
    # TODO (Exercise 2.1): zip results["documents"][0], results["metadatas"][0], and
    # results["distances"][0] together and append one dict per hit to `hits`:
    #   {"source": ..., "chunk_index": ..., "text": ..., "distance": ...}

    return hits

# Try it directly, unwrapped by any agent — this is a plain Python function call.
for hit in search_documents("What construction types are preferred for property coverage?"):
    print(hit)

&nbsp;

### Variant 2 — `extract_submission_information` (structured output)

Wraps Session 1's `SubmissionExtraction` schema + `generate_structured()` — no new logic, just packaged as a callable tool.

In [ ]:
from pydantic import BaseModel

class SubmissionExtraction(BaseModel):
    """Extraction schema for underwriting submissions (same as Session 1)."""
    business_name: str
    industry: str
    locations: list[str]
    requested_limits: str | None = None
    notable_risks: list[str]
    missing_information: list[str]

EXTRACTION_SYSTEM_PROMPT = (
    "Extract underwriting information from the submission text. "
    "For the 'requested_limits' field, if no specific amount is mentioned, set it to null. "
    "For 'notable_risks' and 'missing_information', use empty lists if there are none. "
    "Do not invent values for missing fields."
)

def extract_submission_information(submission_text: str) -> dict:
    """Pull structured facts out of raw submission text."""
    result = client.generate_structured(
        submission_text, SubmissionExtraction, system_prompt=EXTRACTION_SYSTEM_PROMPT
    )
    return result.model_dump()

In [ ]:
sample_submission = (
    "Meridian Cold Storage Partners operates a single refrigerated warehouse facility in "
    "Charleston, SC, located in a FEMA flood zone A. They are requesting $3,000,000 in "
    "property coverage for the building and ammonia-based refrigeration equipment. The "
    "roof was replaced 8 years ago. They have not provided any flood mitigation "
    "documentation. No liability claims have been filed in the past 5 years."
)

extract_submission_information(sample_submission)

&nbsp;

### Variant 3 — `check_underwriting_rules` (deterministic, no LLM call)

Not every tool needs a prompt behind it — a tool is just a described function. This one encodes explicit thresholds straight out of the 4 PDFs (GL guidelines' $5M sign-off line and heavy-machinery questionnaire, the property appetite statement's flood-zone cap and roof-age rule) with plain `if` statements, no model call at all.

In [ ]:
def parse_dollar_amount(text: str | None) -> float | None:
    """Pull the first dollar figure out of a free-text limit string, e.g. '$3,000,000' -> 3000000.0."""
    if not text:
        return None
    match = re.search(r"\$?([\d,]+)", text)
    return float(match.group(1).replace(",", "")) if match else None

📝 **Exercise 2.2** — Below, the GL sign-off, heavy-machinery, and prior-claims rules are already implemented. Add the property appetite statement's flood-zone rule: risks in flood zone `A` or `V` requesting more than $2,000,000 in property coverage *without* flood mitigation documentation on file are outside standard appetite.

In [ ]:
from pydantic import Field

class RuleCheckInput(BaseModel):
    industry: str = Field(description="The business's primary industry")
    requested_limit: float | None = Field(default=None, description="Requested coverage limit in dollars")
    flood_zone: str | None = Field(default=None, description="FEMA flood zone letter, e.g. 'A' or 'V', if applicable")
    flood_mitigation_on_file: bool = Field(default=False, description="Whether flood mitigation documentation is on file")
    roof_age_years: int | None = Field(default=None, description="Age of the roof in years, if known")
    heavy_machinery: bool = Field(default=False, description="Whether the business operates heavy machinery (forklifts, industrial saws, commercial mowing equipment)")
    prior_liability_claims_5yr: int = Field(default=0, description="Number of liability claims in the past 5 years")

def check_underwriting_rules(
    industry: str,
    requested_limit: float | None = None,
    flood_zone: str | None = None,
    flood_mitigation_on_file: bool = False,
    roof_age_years: int | None = None,
    heavy_machinery: bool = False,
    prior_liability_claims_5yr: int = 0,
) -> dict:
    """Check known facts about a risk against explicit underwriting thresholds. No LLM call."""
    flags = []

    if requested_limit is not None and requested_limit > 5_000_000:
        flags.append("Requested limit exceeds standard $5,000,000 max — requires senior underwriter sign-off")

    # TODO (Exercise 2.2): flood-zone cap — flood_zone in ("A", "V") and requested_limit
    # over $2,000,000 without flood_mitigation_on_file should add a flag here.

    if heavy_machinery:
        flags.append("Heavy machinery present — supplemental safety questionnaire required before binding")

    if roof_age_years is not None and roof_age_years > 20:
        flags.append("Roof older than 20 years — current roof condition report required before binding")

    if prior_liability_claims_5yr >= 2:
        flags.append("2+ liability claims in past 5 years — outside preferred risk criteria")

    return {"flags": flags, "within_appetite": len(flags) == 0}

# Try it directly against the Meridian Cold Storage facts above
check_underwriting_rules(
    industry="Cold Storage",
    requested_limit=3_000_000,
    flood_zone="A",
    flood_mitigation_on_file=False,
    roof_age_years=8,
    heavy_machinery=False,
    prior_liability_claims_5yr=0,
)

&nbsp;

### Defining tool schemas

Each tool needs a name, description, and input schema the model can read. We use the same Pydantic mechanism Session 1 used for structured outputs — `model.model_json_schema()` — to generate the schema instead of writing raw JSON Schema by hand.

In [ ]:
class SearchDocumentsInput(BaseModel):
    query: str = Field(description="Natural-language question to search the underwriting guideline manuals for")
    k: int = Field(default=3, description="Number of matching excerpts to return")

class ExtractSubmissionInput(BaseModel):
    submission_text: str = Field(description="Full raw text of the underwriting submission to extract structured facts from")

TOOLS = [
    {
        "name": "search_documents",
        "description": (
            "Search the underwriting guideline manuals (general liability, property "
            "appetite, cyber exclusions, workers' comp classification) for text relevant "
            "to a natural-language question. Returns the top-k most relevant excerpts "
            "with their source document."
        ),
        "input_schema": SearchDocumentsInput.model_json_schema(),
    },
    {
        "name": "extract_submission_information",
        "description": (
            "Extract structured underwriting facts (business name, industry, locations, "
            "requested limits, notable risks, missing information) from the raw text of "
            "a new business submission."
        ),
        "input_schema": ExtractSubmissionInput.model_json_schema(),
    },
    {
        "name": "check_underwriting_rules",
        "description": (
            "Deterministically check known facts about a risk (requested limit, flood "
            "zone, roof age, heavy machinery, prior claims) against the underwriting "
            "manuals' explicit thresholds. Does not use an LLM — returns any rule "
            "violations found."
        ),
        "input_schema": RuleCheckInput.model_json_schema(),
    },
]

TOOL_FUNCTIONS = {
    "search_documents": search_documents,
    "extract_submission_information": extract_submission_information,
    "check_underwriting_rules": check_underwriting_rules,
}

for tool in TOOLS:
    print(tool["name"], "-", tool["description"][:60] + "...")

&nbsp;

💡 **Key Takeaway** — A good tool description is the interface contract between your code and the model's reasoning — vague descriptions produce vague tool selection later.

---

## Section 3 — Building the agent

**Goal:** See the same 3 tools used two ways — a fixed pipeline vs. a model deciding — and build the loop that lets the model decide.

### Guided Demo — the deterministic pipeline

This is the "workflow" half of the workflow-vs-agent comparison: the exact same 3 tools, called in a fixed order *you* chose, with no model in the loop deciding anything.

In [ ]:
def run_fixed_pipeline(submission_text: str) -> str:
    extraction = extract_submission_information(submission_text)
    hits = search_documents(f"underwriting guidelines for {extraction['industry']}")
    rules = check_underwriting_rules(
        industry=extraction["industry"],
        requested_limit=parse_dollar_amount(extraction.get("requested_limits")),
    )

    context = "\n\n".join(hit["text"] for hit in hits)
    prompt = (
        f"Submission facts: {extraction}\n\n"
        f"Relevant guideline excerpts:\n{context}\n\n"
        f"Rule check results: {rules}\n\n"
        "Based only on the above, is this submission within appetite? Explain briefly."
    )
    return client.generate(prompt, system_prompt="You are an underwriting assistant.").text

print(run_fixed_pipeline(sample_submission))

&nbsp;

### The agent loop — same tools, model decides

Now contrast with `generate_with_tools()`: the model sees all 3 tool specs and decides for itself which to call, with what arguments, and when to stop. Each turn, we execute whatever tool it requested, print the raw request and result, append the result to the conversation, and call again — until the model stops requesting tools.

In [ ]:
import json

AGENT_SYSTEM_PROMPT = (
    "You are an underwriting assistant. You have three tools available: "
    "search_documents (search the underwriting guideline manuals), "
    "extract_submission_information (pull structured facts out of raw submission text), "
    "and check_underwriting_rules (check known facts against explicit underwriting thresholds). "
    "Given a new business submission, decide which tools you need and in what order. "
    "Gather enough information to determine whether the risk is within appetite before answering."
)

def run_agent_loop(user_message: str, system_prompt: str, tools: list[dict] = TOOLS, max_iterations: int = 6):
    response = client.generate_with_tools(user_message, tools, system_prompt=system_prompt)
    tool_call_log = []
    iterations = 0

    while response.tool_calls:
        # TODO (Exercise 3.1): stop the loop once `iterations` reaches `max_iterations` -
        # e.g. `if iterations >= max_iterations: break` (or raise, if you'd rather fail loudly
        # than silently return a partial result).

        messages = response.messages
        for call in response.tool_calls:
            result = TOOL_FUNCTIONS[call.name](**call.arguments)
            print(f"  [{iterations}] {call.name}({call.arguments}) -> {result}")
            tool_call_log.append({"name": call.name, "arguments": call.arguments, "result": result})
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

        response = client.generate_with_tools(user_message, tools, system_prompt=system_prompt, messages=messages)
        iterations += 1

    return response, tool_call_log

📝 **Exercise 3.1** — Add the max-iteration guard called out in the `TODO` above, then run the agent below on the same submission used in the fixed pipeline. Compare: which tools did it call, in what order, and how many times — versus the fixed pipeline's Extract → Search → Check Rules → Generate?

In [ ]:
user_message = f"Process this underwriting submission and decide whether it is within appetite:\n\n{sample_submission}"
response, tool_call_log = run_agent_loop(user_message, AGENT_SYSTEM_PROMPT)

print("\n--- Final response ---")
print(response.text)

&nbsp;

💬 **Reflection** — When would you want the fixed pipeline instead of the agent? (Think about predictability, cost, and latency vs. flexibility.)

💡 **Key Takeaway** — A workflow calls tools in an order *you* chose; an agent calls tools in an order *it* chose — same tools, different control flow, different guarantees.

---

## Section 4 — Process a complete submission

**Goal:** Run the agent end-to-end on a brand-new, unseen submission and produce a structured underwriting decision.

Once the agent loop stops, we assemble everything it gathered and call `generate_structured()` with an `UnderwritingDecision` model — turning free-form tool exploration into a predictable, structured decision downstream code can actually use.

In [ ]:
class UnderwritingDecision(BaseModel):
    """Final structured underwriting appetite decision."""
    decision: str = Field(description="One of: 'accept', 'decline', or 'refer to senior underwriter'")
    confidence: float = Field(description="Confidence in this decision, from 0.0 to 1.0")
    risk_factors: list[str] = Field(description="Specific risk factors identified from the gathered tool outputs")
    missing_information: list[str] = Field(description="Information that would be needed to be more certain")
    evidence: list[str] = Field(description="Specific tool outputs (guideline excerpts, rule flags, extracted facts) that support this decision")
    rationale: str = Field(description="Brief explanation tying the decision to the evidence above")

def run_full_agent(submission_text: str) -> tuple[UnderwritingDecision, list[dict]]:
    user_message = (
        f"Process this underwriting submission and determine whether it is within appetite:\n\n{submission_text}"
    )
    response, tool_call_log = run_agent_loop(user_message, AGENT_SYSTEM_PROMPT)

    tool_output_summary = "\n\n".join(
        f"Tool: {entry['name']}({entry['arguments']})\nResult: {entry['result']}"
        for entry in tool_call_log
    )
    decision_prompt = (
        f"Submission:\n{submission_text}\n\n"
        f"Tool outputs gathered while investigating this submission:\n{tool_output_summary}\n\n"
        "Using ONLY the information above, produce a final underwriting appetite decision. "
        "Every risk_factor and evidence entry must trace back to a specific tool output above — "
        "do not introduce new facts."
    )
    decision = client.generate_structured(decision_prompt, UnderwritingDecision, system_prompt=AGENT_SYSTEM_PROMPT)
    return decision, tool_call_log

### Guided Demo — two new, unseen submissions

In [ ]:
NEW_SUBMISSIONS = {
    "SUB-A": (
        "Ironclad Metal Fabrication Co. operates a single shop in Toledo, OH, cutting and "
        "welding structural steel components. They are requesting $6,000,000 in general "
        "liability coverage. Operations include industrial saws and forklifts throughout "
        "the shop floor. They have had 3 liability claims in the past 5 years, primarily "
        "related to equipment incidents."
    ),
    "SUB-B": (
        "Sunrise Boutique Hotel Group is renovating a 67-year-old building in Savannah, GA "
        "into a 40-room boutique hotel, requesting $1,800,000 in property coverage. The "
        "building is currently mid-renovation. The roof was last replaced 25 years ago. No "
        "flood zone was specified in the submission."
    ),
}

📝 **Exercise 4.1** — Run the full agent on both submissions below. For each one, inspect every tool call/result in sequence, then the final structured decision — and identify which tool output(s) informed which `evidence` / `risk_factors` entries.

In [ ]:
print("=== SUB-A ===")
decision_a, tool_call_log_a = run_full_agent(NEW_SUBMISSIONS["SUB-A"])
print("\n--- Decision ---")
print(decision_a.model_dump_json(indent=2))

In [ ]:
print("=== SUB-B ===")
decision_b, tool_call_log_b = run_full_agent(NEW_SUBMISSIONS["SUB-B"])
print("\n--- Decision ---")
print(decision_b.model_dump_json(indent=2))

&nbsp;

💡 **Key Takeaway** — The final decision is only as grounded as the tool outputs it's built from — every field in the structured decision should trace back to a specific tool call, not a guess.

---

## Section 5 — Improve the agent & failure modes

**Goal:** See how small changes to tool descriptions or instructions change agent behavior, and recognize when the agent gets it wrong.

### Variant 1 — baseline (triggers a failure)

Ask about something none of the 4 documents cover: a crime/employee-theft sublimit. This echoes Session 2's cyber-exclusions hallucination demo, but now through the tool lens — the agent has `search_documents` available and will call it, but the retrieved excerpts won't actually answer the question.

In [ ]:
off_topic_submission = (
    "Request from Palmetto Boardwalk Arcade in Myrtle Beach, SC for general liability "
    "coverage. Separately, the owner wants to know: what is our per-occurrence sublimit "
    "for employee theft / crime coverage?"
)

decision, tool_call_log = run_full_agent(off_topic_submission)
print(decision.model_dump_json(indent=2))

📝 **Exercise 5.1** — Diagnose the failure above: did the agent call the wrong tool, pass bad arguments, ignore evidence, or invent a number despite `search_documents` returning nothing relevant? Which pipeline stage — tool description, system prompt, or the final decision prompt — is responsible?

### Variant 2 — rewritten tool description + system instructions

Two changes: `search_documents`'s description now states its scope explicitly, and the system prompt instructs the model to say so when nothing relevant is found rather than guessing.

In [ ]:
TOOLS_V2 = [dict(t) for t in TOOLS]
TOOLS_V2[0] = {
    **TOOLS_V2[0],
    "description": (
        "Search ONLY the 4 underwriting guideline manuals this company has on file: general "
        "liability, property appetite, cyber exclusions, and workers' comp classification. "
        "If a question falls outside these 4 topics, the results will not contain an answer — "
        "do not use them to construct one anyway."
    ),
}

AGENT_SYSTEM_PROMPT_V2 = AGENT_SYSTEM_PROMPT + (
    " If the tool results don't contain the information needed to answer part of a request, "
    "say so explicitly in missing_information rather than guessing or inventing a figure."
)

def run_full_agent_v2(submission_text: str) -> tuple[UnderwritingDecision, list[dict]]:
    user_message = (
        f"Process this underwriting submission and determine whether it is within appetite:\n\n{submission_text}"
    )
    response, tool_call_log = run_agent_loop(user_message, AGENT_SYSTEM_PROMPT_V2, tools=TOOLS_V2)

    tool_output_summary = "\n\n".join(
        f"Tool: {entry['name']}({entry['arguments']})\nResult: {entry['result']}"
        for entry in tool_call_log
    )
    decision_prompt = (
        f"Submission:\n{submission_text}\n\n"
        f"Tool outputs gathered while investigating this submission:\n{tool_output_summary}\n\n"
        "Using ONLY the information above, produce a final underwriting appetite decision. "
        "Every risk_factor and evidence entry must trace back to a specific tool output above — "
        "do not introduce new facts."
    )
    decision = client.generate_structured(decision_prompt, UnderwritingDecision, system_prompt=AGENT_SYSTEM_PROMPT_V2)
    return decision, tool_call_log

decision_v2, tool_call_log_v2 = run_full_agent_v2(off_topic_submission)
print(decision_v2.model_dump_json(indent=2))

📝 Confirm the change: does `missing_information` now call out the crime/theft question explicitly instead of an invented sublimit?

*Note:* this same steering principle — description and instructions shape behavior, not just the model — is also what the `max_iterations` guard from Section 3 protects against on the other failure axis: a tool description or instruction ambiguous enough that the model keeps re-calling the same tool instead of stopping.

&nbsp;

💡 **Key Takeaway** — Agent behavior is steered by descriptions, schemas, and guardrails, not just the underlying model — and every failure mode has a specific, diagnosable cause.

---

## Section 6 — Wrap-up

**Recap:**
- Tools are just existing functions with a name, description, and input schema — not a new kind of code
- An agent is a loop that lets the model choose which tools to call, with what arguments, and when to stop
- A workflow (fixed pipeline) and an agent (loop) can use the exact same tools with completely different guarantees
- The final decision is only as good as the tool outputs behind it — every field should trace back to specific evidence
- Agent behavior is steered by descriptions, schemas, and guardrails, and every failure mode has a diagnosable cause

**Next up — Session 4:** evaluating tool selection, retrieval quality, decision quality, groundedness, and reliability — i.e., how do we know today's agent is actually *good*?

Building an agent was the easy part. Session 4 is about proving it works.